In [ ]:
%load_ext autoreload 
%autoreload 2
%matplotlib inline
import sys
import cProfile #for checking the nr of calls and execution time
import pstats
from pstats import SortKey
from scipy.stats import gaussian_kde
import json
import os
import re
import numpy as np
import multiple_planets_gas_acc as code_gas
from functions_pebble_accretion import *
from functions import *
import functions_plotting as plot
import sim_loader as sim_load
import functions_plotting as plot
import matplotlib.pyplot as plt
import matplotlib as mpl
import astropy.units as u
import pandas as pd
from matplotlib.ticker import ScalarFormatter, LogFormatter, LogLocator, MultipleLocator, AutoMinorLocator
from matplotlib import cm, ticker
from matplotlib import colors
import matplotlib.gridspec as gridspec
import matplotlib.patches as patch
from matplotlib.offsetbox import AnchoredText
from matplotlib.patches import Patch
import matplotlib.lines as mlines 
from matplotlib.animation import FuncAnimation
from matplotlib import pyplot as plt, ticker as mticker
import sim_loader as sim_load
import h5py
import seaborn as sns
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerTuple
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
    "axes.labelsize": 15,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.direction": "in",
    "ytick.direction": "in",
    # major ticks
    "xtick.major.size": 12,
    "ytick.major.size": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    # minor ticks
    "xtick.minor.size": 8,
    "ytick.minor.size": 8,
    "xtick.minor.visible": True,   # minor ticks are hidden by default — need this to actually see them
    "ytick.minor.visible": True,

})

In [ ]:
# ============================================================
# FILE SORTING
# ============================================================

def extract_initial_time(filename):
    with h5py.File(filename, 'r') as f:
        time = f['simulation']['time'][()]
        if np.ndim(time) > 0:
            return float(time[0])
        return float(time)


# ============================================================
# LOAD SIMULATIONS 
# ============================================================
folder_path_multi = ["sims/opacities/1Msun/photoevap/Mdot_scatter"]

simulations_multi = []
sim_parameters_multi = []
parameters_multi = []

for folder in folder_path_multi:
    h5_files = sorted(
        [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".h5")],
        key=extract_initial_time
    )
    simulations_multi.append([sim_load.HDF5toSimRes(f) for f in h5_files])
    sim_parameters_multi.append([sim_load.load_sim_params(f) for f in h5_files])
    parameters_multi.append([sim_load.load_params(f) for f in h5_files])

In [ ]:
alpha_transp = 0.2
print("Number of simulations loaded: ", len(simulations_multi[0]))
c = plt.cm.inferno(np.linspace(0, 1, len(simulations_multi[0])))
print(len(c))
fig, axs = plt.subplots(1, 1, figsize=(6, 6))

for i, folder in enumerate(folder_path_multi):
    for j, (sim, params, sim_params) in enumerate(zip(simulations_multi[i], parameters_multi[i], sim_parameters_multi[i])):
        col = c[j]
        for p in range(sim_params.nr_planets):
            idx = plot.idxs(sim.time.value, sim.mass[p].value, sim.position[p].value, sim.filter_fraction[p],
                             sim.dR_dt[p].value, sim.dM_dt[p].value, params, True)
            axs.scatter(sim.position[p, -1], sim.mass[p, -1], color=col)

## aesthetics of the plot
a_p0 = np.geomspace(0.01, 100, 100)
axs.loglog(a_p0, M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_in, params), params), params), color = "slateblue", linestyle =':', zorder = 0)
axs.loglog(a_p0, M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_fin, params), params), params), color = "slateblue", linestyle =':', zorder = 0)
axs.fill_between(a_p0, M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_fin, params), params), params), M_peb_iso(H_R(a_p0, M_dot_star(sim_params.t_in, params), params), params),  color='slateblue', alpha=0.1)   

axs.set_xlabel('r [AU]', fontsize=25, labelpad=20)
axs.set_xlim(0.01, 1e2)
axs.set_ylim(1e-3, 1e4)
axs.axhline(13*u.M_jup.to(u.M_earth), color = 'grey', linestyle = '--', alpha = 0.5)
axs.set_ylabel('M [$M_{\oplus}$]', fontsize=25, labelpad=20)
axs.set_title("Single planets", fontsize=15, pad=20)


In [ ]:
# color scheme definition
kept_groups = ['super_earth', 'giant']
color_map = {'super_earth': plt.cm.inferno(0.0), 'giant': plt.cm.inferno(0.85)}

planet_records = planet_type_counter(simulations_multi[0], parameters_multi[0], sim_parameters_multi[0], per_planet=True)
df = pd.DataFrame(planet_records)
total_planets = len(df)

kappa_converted = (params.kappa * (u.au**2 / u.M_earth)).to(u.m**2 / u.kg)
kappa_2d = f"{kappa_converted.value:.2f}" if hasattr(kappa_converted, 'value') else f"{float(kappa_converted):.2f}"
kappa_str = str(params.kappa * (u.au**2 / u.M_earth).to(u.m**2 / u.kg))

fig, ax = plt.subplots(1, 1, figsize=(9, 6))
bins = 30
log_bins_range = (-2, 2)

bin_edges = np.logspace(*log_bins_range, bins)
bin_centers = np.sqrt(bin_edges[:-1] * bin_edges[1:])  # geometric mean, correct for log bins

for gtype in kept_groups:
    subset = df.loc[df['group'] == gtype, 'planet_position']
    if not subset.empty:
        counts, _ = np.histogram(subset, bins=bin_edges)
        occurrence_rate = counts / total_planets
        ax.plot(bin_centers, occurrence_rate, marker='o', linewidth=2, label=gtype, color=color_map[gtype])

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('$a_{\\rm p}$ [AU]')
ax.set_ylabel('Occurrence rate')
ax.set_title('singles $\\kappa = ' + kappa_2d + ' \\rm m^2/kg$')
ax.legend()
plt.savefig(f"figures/occurrence/photoevap_singles{kappa_str}_kappa_occurrence.png", bbox_inches='tight')

fig, ax = plt.subplots(1, 1, figsize=(9, 6))
for gtype in kept_groups:
    subset = df.loc[df['group'] == gtype]
    if not subset.empty:
        ax.scatter(subset['planet_position'], subset['planet_mass'], s=20, alpha=0.7, label=gtype, color=color_map[gtype])

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('$a_{\\rm p}$ [AU]')
ax.set_ylabel('$M_{\\rm p}$ [$M_\\oplus$]')
ax.set_title('singles $\\kappa = ' + kappa_2d + ' \\rm m^2/kg$')
ax.legend()
plt.savefig(f"figures/occurrence/photoevap_singles{kappa_str}_kappa_mass_position.png", bbox_inches='tight')

In [ ]:
from scipy.ndimage import gaussian_filter1d

# color scheme definition
kept_groups = ['super_earth', 'giant']
color_map = {'super_earth': plt.cm.inferno(0.0), 'giant': plt.cm.inferno(0.85)}

planet_records = planet_type_counter(simulations_multi[0], parameters_multi[0], sim_parameters_multi[0], per_planet=True)
df = pd.DataFrame(planet_records)
total_planets = len(df)

kappa_converted = (params.kappa * (u.au**2 / u.M_earth)).to(u.m**2 / u.kg)
kappa_2d = f"{kappa_converted.value:.2f}" if hasattr(kappa_converted, 'value') else f"{float(kappa_converted):.2f}"
kappa_str = str(params.kappa * (u.au**2 / u.M_earth).to(u.m**2 / u.kg))

fig, ax = plt.subplots(1, 1, figsize=(9, 6))
bins = 50
log_bins_range = (-2, 2)
smoothing_sigma = 1.5  # in units of bins; increase for more smoothing

bin_edges = np.logspace(*log_bins_range, bins)
bin_centers = np.sqrt(bin_edges[:-1] * bin_edges[1:])  # geometric mean, correct for log bins

for gtype in kept_groups:
    subset = df.loc[df['group'] == gtype, 'planet_position']
    if not subset.empty:
        counts, _ = np.histogram(subset, bins=bin_edges)
        occurrence_pct = 100 * counts / total_planets
        occurrence_pct_smooth = gaussian_filter1d(occurrence_pct, sigma=smoothing_sigma)
        ax.plot(bin_centers, occurrence_pct_smooth, linewidth=2, label=gtype, color=color_map[gtype])

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('$a_{\\rm p}$ [AU]')
ax.set_ylabel('Occurrence rate [%]')
ax.set_title('singles $\\kappa = ' + kappa_2d + ' \\rm m^2/kg$')
ax.legend()
plt.savefig(f"figures/occurrence/photoevap_singles{kappa_str}_kappa_occurrence)smoothed.png", bbox_inches='tight')

fig, ax = plt.subplots(1, 1, figsize=(9, 6))
for gtype in kept_groups:
    subset = df.loc[df['group'] == gtype]
    if not subset.empty:
        ax.scatter(subset['planet_position'], subset['planet_mass'], s=20, alpha=0.7, label=gtype, color=color_map[gtype])

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('$a_{\\rm p}$ [AU]')
ax.set_ylabel('$M_{\\rm p}$ [$M_\\oplus$]')
ax.set_title('singles $\\kappa = ' + kappa_2d + ' \\rm m^2/kg$')
ax.legend()
plt.savefig(f"figures/occurrence/photoevap_singles{kappa_str}_kappa_mass_position.png", bbox_inches='tight')

In [ ]:
from scipy.ndimage import gaussian_filter1d

# color scheme definition
kept_groups = ['super_earth', 'giant']
color_map = {'super_earth': plt.cm.inferno(0.0), 'giant': plt.cm.inferno(0.85)}

planet_records = planet_type_counter(simulations_multi[0], parameters_multi[0], sim_parameters_multi[0], per_planet=True)
df = pd.DataFrame(planet_records)
total_planets = len(df)

kappa_converted = (params.kappa * (u.au**2 / u.M_earth)).to(u.m**2 / u.kg)
kappa_2d = f"{kappa_converted.value:.2f}" if hasattr(kappa_converted, 'value') else f"{float(kappa_converted):.2f}"
kappa_str = str(params.kappa * (u.au**2 / u.M_earth).to(u.m**2 / u.kg))

bins = 50
log_bins_range = (-2, 2)
smoothing_sigma = 1.5  # in units of bins; keep small to avoid erasing real features

bin_edges = np.logspace(*log_bins_range, bins)
bin_centers = np.sqrt(bin_edges[:-1] * bin_edges[1:])  # geometric mean, correct for log bins

fig, ax = plt.subplots(1, 1, figsize=(10, 7))

for gtype in kept_groups:
    subset = df.loc[df['group'] == gtype, 'planet_position']
    if subset.empty:
        continue

    counts, _ = np.histogram(subset, bins=bin_edges)
    occurrence_pct = 100 * counts / total_planets
    occurrence_pct_err = 100 * np.sqrt(counts) / total_planets  # Poisson error, sigma = sqrt(N)
    occurrence_pct_smooth = gaussian_filter1d(occurrence_pct, sigma=smoothing_sigma)

    color = color_map[gtype]

    # raw histogram, as a step curve
    ax.step(bin_centers, occurrence_pct, where='mid', linewidth=1, alpha=0.4,
             color=color, label=f'{gtype} (raw)')

    # # poisson error bars on the raw values
    # ax.errorbar(bin_centers, occurrence_pct, yerr=occurrence_pct_err, fmt='o',
    #             markersize=3, alpha=0.5, color=color, capsize=2, linewidth=1)

    # smoothed curve, solid, on top
    ax.plot(bin_centers, occurrence_pct_smooth, linewidth=2.5,
            color=color, label=f'{gtype} (smoothed, $\\sigma$={smoothing_sigma})')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('$a_{\\rm p}$ [AU]')
ax.set_ylabel('Occurrence rate [%]')
ax.set_title('singles $\\kappa = ' + kappa_2d + ' \\rm m^2/kg$')
ax.legend(fontsize=9)
plt.savefig(f"figures/occurrence/photoevap_singles{kappa_str}_kappa_occurrence_combined.png", bbox_inches='tight')